In [0]:
%run ../../src/utils/config

In [0]:
%python
import os

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

config = load_config(config_path)

In [0]:
%python
# Read catalog and schema names from the existing config
catalog = config["target"]["catalog"]
silver_schema = config["target"]["silver_schema"]
gold_schema = config["target"]["gold_schema"]

gold_dimensions = config["gold"]["dimensions"]

print(f"Catalog      : {catalog}")
print(f"Silver schema: {silver_schema}")
print(f"Gold schema  : {gold_schema}")
print(f"Dimensions   : {len(gold_dimensions)}")

for dim_name, settings in gold_dimensions.items():
    source_table = settings["source_table"]
    natural_keys = settings["natural_key"]
    strategy = settings["load_strategy"]

    print(
        f"{dim_name}: "
        f"source={catalog}.{silver_schema}.{source_table}, "
        f"keys={natural_keys}, "
        f"strategy={strategy}"
    )

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def prepare_dimension_source(dim_name, settings):
    source_table = (
        f"{catalog}.{silver_schema}.{settings['source_table']}"
    )

    natural_keys = settings["natural_key"]

    attributes = settings.get(
        "tracked_columns",
        settings.get("attributes", [])
    )

    selected_columns = list(dict.fromkeys(natural_keys + attributes))

    # Read Silver and select the columns needed by this dimension
    source_df = (
        spark.table(source_table)
        .select(*selected_columns,
                *[
                    F.col(c)
                    for c in ["silver_processed_at",
                              "source_ingestion_timestamp"]
                    if c in spark.table(source_table).columns
                    and c not in selected_columns
                ])
    )

    source_count = source_df.count()

    # Natural keys must not be null
    valid_df = source_df
    for key in natural_keys:
        valid_df = valid_df.filter(F.col(key).isNotNull())

    valid_count = valid_df.count()

    # Prefer the most recently processed Silver record
    order_columns = [
        F.col(c).desc_nulls_last()
        for c in [
            "silver_processed_at",
            "source_ingestion_timestamp"
        ]
        if c in valid_df.columns
    ]

    if order_columns:
        window_spec = (
            Window.partitionBy(*natural_keys)
            .orderBy(*order_columns)
        )

        prepared_df = (
            valid_df
            .withColumn("_rn", F.row_number().over(window_spec))
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )
    else:
        prepared_df = valid_df.dropDuplicates(natural_keys)

    # Keep only dimension columns in the final result
    prepared_df = prepared_df.select(*selected_columns)

    print(f"Dimension: {dim_name}")
    print(f"  Source rows       : {source_count}")
    print(f"  Rows with valid keys: {valid_count}")
    print(f"  Prepared rows     : {prepared_df.count()}")

    return prepared_df

In [0]:
%python
dimension_sources = {}

for dim_name, settings in gold_dimensions.items():
    dimension_sources[dim_name] = prepare_dimension_source(
        dim_name,
        settings
    )

    print("-" * 60)

In [0]:
%python
for dim_name in gold_dimensions:
    target_table = f"{catalog}.{gold_schema}.{dim_name}"

    row_count = spark.table(target_table).count()

    print(f"{dim_name}: {row_count} rows")

In [0]:
%python
from pyspark.sql import functions as F

settings = gold_dimensions["dim_pokemon"]
pokemon_source = dimension_sources["dim_pokemon"]

tracked_columns = settings["tracked_columns"]

# Hash the attributes that should trigger a new SCD2 version
pokemon_source = pokemon_source.withColumn(
    "attribute_hash",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(c) for c in tracked_columns])
        ),
        256
    )
)

pokemon_source.createOrReplaceTempView("src_dim_pokemon")

In [0]:
%python
from pyspark.sql import functions as F

target_table = f"{catalog}.{gold_schema}.dim_pokemon"

# 1. Expire current versions whose tracked attributes changed
spark.sql(f"""
    MERGE INTO {target_table} AS tgt
    USING src_dim_pokemon AS src
       ON tgt.pokemon_id = src.pokemon_id
      AND tgt.is_current = TRUE

    WHEN MATCHED
     AND NOT (tgt.attribute_hash <=> src.attribute_hash)
    THEN UPDATE SET
        tgt.effective_to = current_timestamp(),
        tgt.is_current = FALSE
""")

# 2. Insert new Pokémon and new versions of changed Pokémon
spark.sql(f"""
    INSERT INTO {target_table} (
        pokemon_id,
        name,
        base_experience,
        height,
        weight,
        is_default,
        pokemon_order,
        species_name,
        species_url,
        effective_from,
        effective_to,
        is_current,
        attribute_hash,
        gold_created_at
    )
    SELECT
        src.pokemon_id,
        src.name,
        src.base_experience,
        src.height,
        src.weight,
        src.is_default,
        src.pokemon_order,
        src.species_name,
        src.species_url,
        current_timestamp(),
        CAST(NULL AS TIMESTAMP),
        TRUE,
        src.attribute_hash,
        current_timestamp()
    FROM src_dim_pokemon AS src
    LEFT ANTI JOIN {target_table} AS tgt
      ON tgt.pokemon_id = src.pokemon_id
     AND tgt.is_current = TRUE
""")

In [0]:
%python
target_table = f"{catalog}.{gold_schema}.dim_pokemon"

validation = spark.sql(f"""
    SELECT
        COUNT(*) AS total_versions,
        COUNT(DISTINCT pokemon_id) AS distinct_pokemon,
        SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_versions,
        SUM(CASE WHEN pokemon_key IS NULL THEN 1 ELSE 0 END) AS null_keys,
        SUM(CASE WHEN is_current AND effective_to IS NOT NULL
                 THEN 1 ELSE 0 END) AS invalid_current_rows,
        SUM(CASE WHEN NOT is_current AND effective_to IS NULL
                 THEN 1 ELSE 0 END) AS invalid_historical_rows
    FROM {target_table}
""").first()

print(validation.asDict())

if validation["null_keys"] > 0:
    raise ValueError("dim_pokemon validation failed: null pokemon_key found.")

if validation["invalid_current_rows"] > 0:
    raise ValueError(
        "dim_pokemon validation failed: current records must have "
        "effective_to = NULL."
    )

if validation["invalid_historical_rows"] > 0:
    raise ValueError(
        "dim_pokemon validation failed: historical records must have "
        "effective_to populated."
    )

if validation["current_versions"] != validation["distinct_pokemon"]:
    raise ValueError(
        "dim_pokemon validation failed: each Pokémon must have "
        "exactly one current version."
    )

print("dim_pokemon validation PASSED")

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def upsert_type1_dimension(dim_name, settings):
    target_table = f"{catalog}.{gold_schema}.{dim_name}"
    source_df = dimension_sources[dim_name]

    natural_keys = settings["natural_key"]
    attributes = settings.get("attributes", [])

    # Exclude records with null natural keys
    for key in natural_keys:
        source_df = source_df.filter(F.col(key).isNotNull())

    # Fail if the source has duplicate natural keys
    has_duplicates = (
        source_df.groupBy(*natural_keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if has_duplicates:
        raise ValueError(
            f"{dim_name}: duplicate natural keys found in source."
        )

    delta_target = DeltaTable.forName(spark, target_table)

    merge_condition = " AND ".join(
        f"t.`{key}` = s.`{key}`"
        for key in natural_keys
    )

    # Update descriptive attributes only.
    # Preserve the existing surrogate key and creation timestamp.
    update_set = {
        column: f"s.`{column}`"
        for column in attributes
    }

    # Insert natural keys, attributes, and Gold metadata.
    insert_columns = list(
        dict.fromkeys(natural_keys + attributes)
    )

    insert_values = {
        column: f"s.`{column}`"
        for column in insert_columns
    }

    # Fix: gold_created_at is a required Gold metadata column.
    insert_values["gold_created_at"] = "current_timestamp()"

    merge_builder = (
        delta_target.alias("t")
        .merge(source_df.alias("s"), merge_condition)
    )

    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(
            set=update_set
        )

    (
        merge_builder
        .whenNotMatchedInsert(values=insert_values)
        .execute()
    )

    print(f"{dim_name}: Type 1 upsert completed.")

In [0]:
%python
for dim_name, settings in gold_dimensions.items():
    if settings["load_strategy"] == "upsert":
        upsert_type1_dimension(dim_name, settings)

In [0]:
%python
from pyspark.sql import functions as F

for dim_name, settings in gold_dimensions.items():
    target_table = f"{catalog}.{gold_schema}.{dim_name}"
    natural_keys = settings["natural_key"]
    surrogate_key = settings["surrogate_key"]

    df = spark.table(target_table)

    # Check surrogate keys
    null_surrogate_count = (
        df.filter(F.col(surrogate_key).isNull()).count()
    )

    if null_surrogate_count > 0:
        raise ValueError(
            f"{dim_name}: {null_surrogate_count} null surrogate keys found."
        )

    # SCD2 permits multiple historical versions,
    # but only one current row per natural key.
    if settings["load_strategy"] == "scd2":
        check_df = df.filter(F.col("is_current") == True)
    else:
        check_df = df

    duplicate_natural_keys = (
        check_df.groupBy(*natural_keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_natural_keys:
        raise ValueError(
            f"{dim_name}: duplicate natural keys found."
        )

    # Check required creation timestamp
    null_created_count = (
        df.filter(F.col("gold_created_at").isNull()).count()
    )

    if null_created_count > 0:
        raise ValueError(
            f"{dim_name}: null gold_created_at values found."
        )

    print(
        f"{dim_name}: rows={df.count()}, "
        "validation=PASSED"
    )

print("All Gold dimension validations passed.")